# Top models of a study

Shows, for each k, the best configurations of one study next to the daily and 7-day baselines. It computes nothing new: it reads `metrics/regime_metrics.csv`, written by `scripts/06_regime_metrics.py`, so run the script first.

How the configurations are chosen, for each k, with the days of `choose_period` (val) only:
1. Each configuration (model, z_dim, lam, beta, input) is the mean over its seeds, shown as mean +- std. Each input is measured in its matched view: the daily price for unsmoothed inputs, the backward 7-day mean of the price for smoothed ones.
2. Configurations that switch regime more than `persistence_factor` (2) times the switches per year of the 7-day baseline with the same k are left out: they do not make regimes that last weeks or months.
3. The rest are ordered by the F1 of the expensive days and the first `top` are shown.
4. The same configurations are then shown with the days of `report_period` (test), in the same order. The test days are not used to choose.

The El Nino capture and lift of each episode are shown but not used to order: the capture grows with the size of the most expensive regime. Precision, recall and F1 are those of the expensive days in the most expensive regime (a day is expensive when its real price is above the 66.7, 75 or 80 percentile of the days of the same period for k = 3, 4 and 5).

<d.benavidess@uniandes.edu.co>

In [6]:
import sys
sys.path.append("..")

from pathlib import Path

import pandas as pd

from src.metrics import comparison_table, persistence_factor, ranking

## 0. Configuration

`study` is a study folder name of `experiments/` (for example `weekly`, `dimensions`, `lam_beta_grid`). `top` is how many configurations are shown per k. `choose_period` is the period used to filter and order the configurations (val); `report_period` is shown for the same configurations, in the same order, and is not used to choose (test).

In [7]:
study = "daily"
top = 3
k_values = [3, 4, 5]
choose_period, report_period = "val", "test"

## 1. Top configurations

In [8]:
rows = pd.read_csv(Path("../metrics/regime_metrics.csv"))
if study not in set(rows["study"]):
    raise ValueError(f"{study} is not in metrics/regime_metrics.csv: {sorted(set(rows['study']))}")

baseline_labels = {"features": "Baseline", "view": "View"}
model_labels = {"rank_c": "Rank", "z_dim": "z_dim", "lam": "lam", "beta": "beta", "features": "Features",
                "view": "View", "seeds": "Seeds"}
keys = ["model", "study", "z_dim", "lam", "beta", "features", "view"]

for k in k_values:
    stats = ranking(rows, k, choose_period)
    limit = stats["switches_limit"].iloc[0]
    candidates = stats[stats["study"] == study]
    passed = candidates.dropna(subset=["rank_c"]).sort_values("rank_c").head(top)
    passed = passed.assign(rank_c=range(1, len(passed) + 1))
    print(f"k = {k}: {study}, {len(candidates)} configurations, {len(candidates.dropna(subset=['rank_c']))} switch at most "
          f"{persistence_factor:g} x the 7-day baseline in {choose_period} ({limit:.1f} per year)")
    for period in (choose_period, report_period):
        period_stats = stats if period == choose_period else ranking(rows, k, period)
        print(f"Period: {period}" + (" (chooses the models)" if period == choose_period else " (reported)"))
        print(comparison_table(period_stats[period_stats["model"] == "level_only"], baseline_labels))
        if passed.empty:
            print(f"No configuration of {study} experiments passed the switches filter with k = {k}.")
        else:
            shown = passed[keys + ["rank_c"]].merge(period_stats.drop(columns=["rank_c"]), on=keys)
            print(comparison_table(shown.sort_values("rank_c"), model_labels))
    print()

k = 3: daily, 36 configurations, 15 switch at most 2 x the 7-day baseline in val (31.7 per year)
Period: val (chooses the models)
+----------+-------+---------------+-------------------+-----------------+------------+-----------------+------------+-------------+----------+------+
| Baseline |  View | Switches / yr | Top regime days % | Capture 15-16 % | Lift 15-16 | Capture 23-24 % | Lift 23-24 | Precision % | Recall % | F1 % |
+----------+-------+---------------+-------------------+-----------------+------------+-----------------+------------+-------------+----------+------+
|  level   | daily |      62.7     |        53.8       |       93.8      |    1.74    |        -        |     -      |     60.9    |   98.4   | 75.3 |
| level 7d |   7d  |      15.8     |        53.0       |       93.3      |    1.76    |        -        |     -      |     62.4    |   99.3   | 76.7 |
+----------+-------+---------------+-------------------+-----------------+------------+-----------------+----------

In [9]:
features_shown = None  # None = todas; o "latents + level" para ver solo la entrada sin suavizar
for k in k_values:
    for period in (choose_period, report_period):
        stats = ranking(rows, k, period)
        candidates = stats[stats["study"] == study]
        if features_shown:
            candidates = candidates[candidates["features"] == features_shown]
        print(f"k = {k}, {period}: {study}, all configurations (filter limit {stats['switches_limit'].iloc[0]:.1f} per year)")
        print(comparison_table(candidates.sort_values("expensive_f1_pct_mean", ascending=False), model_labels))

k = 3, val: daily, all configurations (filter limit 31.7 per year)
+------+-------+-------+------+-----------------------+-------+-------+---------------+-------------------+-----------------+--------------+-----------------+------------+--------------+--------------+--------------+
| Rank | z_dim |  lam  | beta |        Features       |  View | Seeds | Switches / yr | Top regime days % | Capture 15-16 % |  Lift 15-16  | Capture 23-24 % | Lift 23-24 | Precision %  |   Recall %   |     F1 %     |
+------+-------+-------+------+-----------------------+-------+-------+---------------+-------------------+-----------------+--------------+-----------------+------------+--------------+--------------+--------------+
| 1.0  |  4.0  |  0.0  | 0.5  | latents 7d + level 7d |   7d  |   3   |  19.0 +- 1.3  |    53.0 +- 0.5    |   93.3 +- 1.1   | 1.76 +- 0.01 |        -        |     -      | 61.8 +- 0.4  | 98.4 +- 0.3  | 75.9 +- 0.2  |
| 2.0  |  4.0  |  0.0  | 0.25 | latents 7d + level 7d |   7d  |  